# 📱 Notebook 01 — Google Play Store Review Scraping

**Project:** Fintech Sentiment Intelligence  
**Sprint:** Week 1 — Data Foundation  
**PBI:** 1.2 — Data Scraping (Day 2)  
**Author:** John Kirima  

---

### Objective

Scrape **5,000–10,000 reviews per app** from the Google Play Store for four major fintech apps:

| App | Package ID | Role |
|-----|-----------|------|
| **Cash App** | `com.squareup.cash` | Primary Subject (rating drop) |
| **Venmo** | `com.venmo` | Benchmark Competitor |
| **Chime** | `com.onedebit.chime` | Benchmark Competitor |
| **PayPal** | `com.paypal.android.p2pmobile` | Benchmark Competitor |

### Output
- `data/raw/fintech_reviews_raw.csv` — Combined raw reviews (all 4 apps)
- Individual per-app CSVs: `data/raw/{app}_raw.csv`

### Library
- [`google-play-scraper`](https://pypi.org/project/google-play-scraper/) — Python wrapper for Google Play Store scraping

---
## 1️⃣ Setup & Imports

In [ ]:
# Install if needed (uncomment to run once)
# !pip install google-play-scraper pandas tqdm

In [ ]:
import os
import time
import datetime

import pandas as pd
from google_play_scraper import Sort, reviews, app
from tqdm import tqdm

print(f"✅ All imports successful")
print(f"📅 Scrape started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2️⃣ App Configurations

Define the 4 fintech apps with their Google Play package IDs.

In [ ]:
# ============================================
# App configurations — Google Play Store
# ============================================

APPS = [
    {
        "app_name": "Cash App",
        "package_id": "com.squareup.cash",
        "file_label": "cashapp"
    },
    {
        "app_name": "Venmo",
        "package_id": "com.venmo",
        "file_label": "venmo"
    },
    {
        "app_name": "Chime",
        "package_id": "com.onedebit.chime",
        "file_label": "chime"
    },
    {
        "app_name": "PayPal",
        "package_id": "com.paypal.android.p2pmobile",
        "file_label": "paypal"
    }
]

# ============================================
# Scraping settings
# ============================================

TARGET_REVIEWS_PER_APP = 10000    # Max reviews to pull per app
BATCH_SIZE = 200                  # Reviews per API call (max 200)
SLEEP_BETWEEN_BATCHES = 2         # Seconds between batch requests
SLEEP_BETWEEN_APPS = 30           # Seconds between apps (rate limit protection)
LANGUAGE = 'en'
COUNTRY = 'us'

print(f"📋 Apps to scrape: {len(APPS)}")
print(f"🎯 Target per app: {TARGET_REVIEWS_PER_APP:,} reviews")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"⏱️  Sleep between batches: {SLEEP_BETWEEN_BATCHES}s")
print(f"⏱️  Sleep between apps: {SLEEP_BETWEEN_APPS}s")
for a in APPS:
    print(f"  • {a['app_name']:12s} → {a['package_id']}")

---
## 3️⃣ Pre-flight Check — Verify App IDs

Confirm each app exists on Google Play before scraping.

In [ ]:
print("🔍 Verifying app IDs on Google Play Store...\n")

for a in APPS:
    try:
        info = app(a['package_id'], lang=LANGUAGE, country=COUNTRY)
        print(f"  ✅ {a['app_name']:12s} | Rating: {info.get('score', 'N/A'):.1f} "
              f"| Reviews: {info.get('reviews', 'N/A'):,} "
              f"| Installs: {info.get('installs', 'N/A')}")
    except Exception as e:
        print(f"  ❌ {a['app_name']:12s} | ERROR: {e}")
        print(f"     → Check package ID: {a['package_id']}")

print("\n✅ Pre-flight check complete.")

---
## 4️⃣ Scraping Function

Core function that:
- Pulls reviews in batches of 200 (API max)
- Uses continuation tokens for pagination
- Sleeps between batches to avoid rate limits
- Has robust error handling with retry logic
- Shows progress via tqdm progress bar

In [ ]:
def scrape_app_reviews(
    package_id: str,
    app_name: str,
    target_count: int = 10000,
    batch_size: int = 200,
    sleep_between: float = 2.0,
    max_retries: int = 3,
    lang: str = 'en',
    country: str = 'us'
) -> pd.DataFrame:
    """
    Scrape Google Play Store reviews for a single app.
    
    Parameters
    ----------
    package_id : str
        Google Play package ID (e.g. 'com.squareup.cash')
    app_name : str
        Human-readable app name for labeling
    target_count : int
        Maximum number of reviews to scrape
    batch_size : int
        Reviews per API call (max 200)
    sleep_between : float
        Seconds to sleep between batch requests
    max_retries : int
        Number of retries on failure before giving up
    lang : str
        Language filter
    country : str
        Country filter
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: app_name, review_id, date, rating,
        review_text, thumbs_up, app_version
    """
    all_reviews = []
    continuation_token = None
    consecutive_failures = 0
    batches_fetched = 0
    
    # Calculate expected batches for progress bar
    expected_batches = (target_count + batch_size - 1) // batch_size
    
    print(f"\n{'='*60}")
    print(f"📱 Scraping: {app_name} ({package_id})")
    print(f"   Target: {target_count:,} reviews | Batch size: {batch_size}")
    print(f"{'='*60}")
    
    pbar = tqdm(total=target_count, desc=f"  {app_name}", unit=" reviews")
    
    while len(all_reviews) < target_count:
        retries = 0
        batch_result = None
        
        # Retry loop for this batch
        while retries < max_retries:
            try:
                batch_result, continuation_token = reviews(
                    package_id,
                    lang=lang,
                    country=country,
                    sort=Sort.NEWEST,
                    count=batch_size,
                    continuation_token=continuation_token
                )
                consecutive_failures = 0  # Reset on success
                break  # Success — exit retry loop
                
            except Exception as e:
                retries += 1
                wait_time = sleep_between * (2 ** retries)  # Exponential backoff
                print(f"\n  ⚠️  Batch error (attempt {retries}/{max_retries}): {e}")
                print(f"     Waiting {wait_time:.0f}s before retry...")
                time.sleep(wait_time)
        
        # If all retries failed
        if batch_result is None:
            consecutive_failures += 1
            print(f"\n  ❌ Batch failed after {max_retries} retries.")
            if consecutive_failures >= 3:
                print(f"  🛑 3 consecutive batch failures. Stopping {app_name}.")
                break
            continue
        
        # If no reviews returned, we've exhausted available reviews
        if not batch_result:
            print(f"\n  ℹ️  No more reviews available. Total: {len(all_reviews):,}")
            break
        
        all_reviews.extend(batch_result)
        batches_fetched += 1
        pbar.update(len(batch_result))
        
        # If continuation token is None, no more pages
        if continuation_token is None:
            print(f"\n  ℹ️  Reached end of available reviews.")
            break
        
        # Rate limit protection
        time.sleep(sleep_between)
    
    pbar.close()
    
    # Convert to DataFrame
    if not all_reviews:
        print(f"  ❌ No reviews scraped for {app_name}!")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_reviews)
    
    # Standardize column names to match project schema
    column_mapping = {
        'reviewId': 'review_id',
        'userName': 'user_name',
        'content': 'review_text',
        'score': 'rating',
        'thumbsUpCount': 'thumbs_up',
        'reviewCreatedVersion': 'app_version',
        'at': 'date',
        'replyContent': 'developer_reply',
        'repliedAt': 'reply_date'
    }
    
    df = df.rename(columns=column_mapping)
    df['app_name'] = app_name
    
    # Select and order the target columns (keep extras as bonus)
    target_cols = ['app_name', 'review_id', 'date', 'rating', 'review_text',
                   'thumbs_up', 'app_version']
    bonus_cols = ['user_name', 'developer_reply', 'reply_date']
    
    # Keep only columns that exist
    keep_cols = [c for c in target_cols + bonus_cols if c in df.columns]
    df = df[keep_cols]
    
    # Summary
    print(f"\n  ✅ {app_name}: {len(df):,} reviews scraped in {batches_fetched} batches")
    if 'date' in df.columns and not df['date'].isna().all():
        print(f"     Date range: {df['date'].min()} → {df['date'].max()}")
    if 'rating' in df.columns:
        print(f"     Avg rating: {df['rating'].mean():.2f}")
        print(f"     Rating dist: {df['rating'].value_counts().sort_index().to_dict()}")
    
    return df

---
## 5️⃣ Execute Scraping — All 4 Apps

This is the main scraping loop. It will:
1. Scrape each app sequentially
2. Save individual per-app CSVs to `data/raw/`
3. Sleep 30 seconds between apps to respect rate limits

⏱️ **Expected time:** ~15–30 minutes total (depends on available reviews & rate limits)

In [ ]:
# Create output directory
RAW_DIR = os.path.join('..', 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
print(f"📁 Output directory: {os.path.abspath(RAW_DIR)}")

# ============================================
# Main scraping loop
# ============================================

all_dfs = []           # Collect DataFrames for all apps
scrape_log = []        # Metadata log
scrape_start = time.time()

for i, app_config in enumerate(APPS):
    app_start = time.time()
    
    # Scrape this app
    df = scrape_app_reviews(
        package_id=app_config['package_id'],
        app_name=app_config['app_name'],
        target_count=TARGET_REVIEWS_PER_APP,
        batch_size=BATCH_SIZE,
        sleep_between=SLEEP_BETWEEN_BATCHES,
        lang=LANGUAGE,
        country=COUNTRY
    )
    
    app_elapsed = time.time() - app_start
    
    if not df.empty:
        # Save individual app CSV
        app_csv_path = os.path.join(RAW_DIR, f"{app_config['file_label']}_raw.csv")
        df.to_csv(app_csv_path, index=False)
        print(f"  💾 Saved: {app_csv_path} ({len(df):,} rows)")
        
        all_dfs.append(df)
        
        # Log metadata
        scrape_log.append({
            'app_name': app_config['app_name'],
            'package_id': app_config['package_id'],
            'reviews_scraped': len(df),
            'date_min': str(df['date'].min()) if 'date' in df.columns else 'N/A',
            'date_max': str(df['date'].max()) if 'date' in df.columns else 'N/A',
            'avg_rating': round(df['rating'].mean(), 2) if 'rating' in df.columns else 'N/A',
            'scrape_time_sec': round(app_elapsed, 1),
            'scrape_date': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
    else:
        print(f"  ⚠️  No data for {app_config['app_name']} — skipping save.")
        scrape_log.append({
            'app_name': app_config['app_name'],
            'package_id': app_config['package_id'],
            'reviews_scraped': 0,
            'error': 'No reviews returned'
        })
    
    # Sleep between apps (skip after last app)
    if i < len(APPS) - 1:
        print(f"\n  ⏳ Sleeping {SLEEP_BETWEEN_APPS}s before next app (rate limit protection)...")
        time.sleep(SLEEP_BETWEEN_APPS)

total_elapsed = time.time() - scrape_start
print(f"\n{'='*60}")
print(f"🏁 All scraping complete! Total time: {total_elapsed/60:.1f} minutes")
print(f"{'='*60}")

---
## 6️⃣ Combine & Save Master Dataset

In [ ]:
if all_dfs:
    # Combine all app DataFrames
    master_df = pd.concat(all_dfs, ignore_index=True)
    
    # Save master CSV
    master_csv_path = os.path.join(RAW_DIR, 'fintech_reviews_raw.csv')
    master_df.to_csv(master_csv_path, index=False)
    
    print(f"💾 Master dataset saved: {master_csv_path}")
    print(f"   Total rows: {len(master_df):,}")
    print(f"   File size: {os.path.getsize(master_csv_path) / (1024*1024):.1f} MB")
    print(f"\n📋 Column list: {list(master_df.columns)}")
    print(f"\n🔍 First 3 rows:")
    display(master_df.head(3))
else:
    print("❌ No data scraped! Check errors above.")

---
## 7️⃣ Detailed Summary & Validation

In [ ]:
if all_dfs:
    print("\n" + "="*70)
    print("📊 SCRAPE SUMMARY REPORT")
    print("="*70)
    
    # --- Total stats ---
    total_reviews = len(master_df)
    print(f"\n🔢 Total Reviews Scraped: {total_reviews:,}")
    target_met = "✅ TARGET MET" if total_reviews >= 20000 else "⚠️  Below 35K target"
    print(f"   Target (35,000): {target_met}")
    
    # --- Per-app breakdown ---
    print(f"\n{'─'*70}")
    print(f"📱 PER-APP BREAKDOWN")
    print(f"{'─'*70}")
    print(f"{'App':<15} {'Count':>8} {'%':>7} {'Avg Rating':>11} {'Date Range':>30}")
    print(f"{'─'*70}")
    
    for app_name in master_df['app_name'].unique():
        app_df = master_df[master_df['app_name'] == app_name]
        count = len(app_df)
        pct = count / total_reviews * 100
        avg_rating = app_df['rating'].mean()
        
        if 'date' in app_df.columns and not app_df['date'].isna().all():
            date_min = pd.to_datetime(app_df['date']).min().strftime('%Y-%m-%d')
            date_max = pd.to_datetime(app_df['date']).max().strftime('%Y-%m-%d')
            date_range = f"{date_min} → {date_max}"
        else:
            date_range = "N/A"
        
        print(f"{app_name:<15} {count:>8,} {pct:>6.1f}% {avg_rating:>10.2f}  {date_range:>30}")
    
    # --- Rating distribution ---
    print(f"\n{'─'*70}")
    print(f"⭐ RATING DISTRIBUTION (ALL APPS)")
    print(f"{'─'*70}")
    
    rating_dist = master_df['rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        pct = count / total_reviews * 100
        bar = '█' * int(pct / 2)
        stars = '⭐' * int(rating)
        print(f"  {stars:<12} ({int(rating)}★): {count:>7,}  ({pct:>5.1f}%)  {bar}")
    
    # --- Per-app rating distribution ---
    print(f"\n{'─'*70}")
    print(f"⭐ RATING DISTRIBUTION PER APP")
    print(f"{'─'*70}")
    
    for app_name in master_df['app_name'].unique():
        app_df = master_df[master_df['app_name'] == app_name]
        dist = app_df['rating'].value_counts().sort_index()
        print(f"\n  {app_name}:")
        for rating, count in dist.items():
            pct = count / len(app_df) * 100
            bar = '█' * int(pct / 2)
            print(f"    {int(rating)}★: {count:>6,}  ({pct:>5.1f}%)  {bar}")
    
    # --- Data quality checks ---
    print(f"\n{'─'*70}")
    print(f"🔍 DATA QUALITY CHECKS")
    print(f"{'─'*70}")
    
    null_counts = master_df.isnull().sum()
    print(f"  Null values per column:")
    for col, nulls in null_counts.items():
        status = "✅" if nulls == 0 else "⚠️"
        print(f"    {status} {col}: {nulls:,} nulls ({nulls/total_reviews*100:.1f}%)")
    
    dupes = master_df.duplicated(subset=['review_id']).sum()
    print(f"\n  Duplicate review IDs: {dupes:,} {'✅' if dupes == 0 else '⚠️  (will clean in notebook 02)'}")
    
    # --- Files saved ---
    print(f"\n{'─'*70}")
    print(f"💾 FILES SAVED")
    print(f"{'─'*70}")
    
    for f in os.listdir(RAW_DIR):
        if f.endswith('.csv'):
            fpath = os.path.join(RAW_DIR, f)
            fsize = os.path.getsize(fpath) / (1024*1024)
            print(f"  📄 {f:<35} ({fsize:.1f} MB)")
    
    print(f"\n{'='*70}")
    print(f"✅ SCRAPING COMPLETE — Ready for Notebook 02 (Cleaning & EDA)")
    print(f"{'='*70}")
else:
    print("❌ No data to summarize.")

---
## 8️⃣ Scrape Metadata Log

Save scraping metadata for documentation and reproducibility.

In [ ]:
# Save scrape metadata
if scrape_log:
    log_df = pd.DataFrame(scrape_log)
    log_path = os.path.join(RAW_DIR, 'scrape_metadata.csv')
    log_df.to_csv(log_path, index=False)
    print(f"📋 Scrape metadata saved: {log_path}")
    display(log_df)
else:
    print("No metadata to save.")

---
## ✅ Next Steps

With raw data in hand, proceed to:

1. **Notebook 02 — Cleaning & EDA** (`02_cleaning_eda.ipynb`)
   - Remove duplicates
   - Handle nulls
   - Clean review text
   - Feature engineering (review_length, year_month, is_negative)
   - Save to `data/clean/all_apps_clean.csv`

### Commit Message
```
feat: raw data scrape complete — X total reviews across 4 apps
```